# SentenceChunker Demo
Shows character-mode and token-mode chunking on a medium-length passage.

## Imports

In [1]:
# Standard Library
# Third Party Library
# Private Library
from cleave.chunker.sentence import SentenceChunker
from cleave.schemas import (
    ChunkParams, ChunkUnit,
    ContentBlock, ContentType,
    Document, DocumentPage, Source, SourceType,
)

## Data

In [2]:
TEXT = """\
Chunking is the process of splitting a long document into smaller, overlapping pieces
so that each piece fits within the context window of a language model. The overlap
ensures that no information is lost at the boundary between two adjacent chunks -
a sentence or phrase that straddles a boundary will appear in both neighbours. Why should 
we apply chunking? It is because without chunking, we explode our context window!

Fixed chunking is the simplest strategy: a sliding window of fixed size moves over
the text with a fixed step. The step is always smaller than the window, which creates
the overlap. The window can be measured in characters or in tokens depending on the
use case. Token-based chunking is more precise for LLM context limits; character-based
chunking is faster and requires no tokeniser.
"""

In [3]:
print(f"Text length : {len(TEXT)} chars")
print(f"Preview     : {TEXT[:80]}...")

Text length : 811 chars
Preview     : Chunking is the process of splitting a long document into smaller, overlapping p...


In [4]:
source = Source(type=SourceType.txt, name="demo.txt", location="/tmp/demo.txt")

In [5]:
def make_document(text: str) -> Document:
    page = DocumentPage(
        page_number=1,
        blocks=[ContentBlock(type=ContentType.text, content=text, position=0)],
    )
    return Document(source=source, pages=[page], total_pages=1)

## Character mode

In [6]:
CHUNK_SIZE = 150
CHUNK_OVERLAP = 20

In [7]:
char_chunker = SentenceChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP))
char_chunks = char_chunker.chunk(make_document(TEXT))

In [8]:
print(f"chunk_size={CHUNK_SIZE}  chunk_overlap={CHUNK_OVERLAP}")
print(f"Total chunks : {len(char_chunks)}\n")

for c in char_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}\u2013{c.char_end:<3}  tokens={c.token_count:>3}  \u2502 {c.text[:60]!r}")

chunk_size=150  chunk_overlap=20
Total chunks : 5

[0] chars   0–156  tokens= 30  │ 'Chunking is the process of splitting a long document into sm'
[1] chars 136–329  tokens= 39  │ 'of a language model. The overlap\nensures that no information'
[2] chars 309–535  tokens= 49  │ ' in both neighbours. Why should \nwe apply chunking? It is be'
[3] chars 515–687  tokens= 38  │ 't with a fixed step. The step is always smaller than the win'
[4] chars 667–687  tokens=  7  │ 'ing on the\nuse case.'


### Overlap inspection
The last `chunk_overlap` characters of chunk *n* should appear at the start of chunk *n+1*.

In [9]:
for a, b in zip(char_chunks, char_chunks[1:]):
    tail = a.text[-CHUNK_OVERLAP:]
    head = b.text[:CHUNK_OVERLAP]
    match = "\u2713" if tail == head else "\u2717"
    print(f"chunk {a.index}\u2192{b.index}  {match}  overlap: {tail!r}")

chunk 0→1  ✓  overlap: 'of a language model.'
chunk 1→2  ✓  overlap: ' in both neighbours.'
chunk 2→3  ✓  overlap: 't with a fixed step.'
chunk 3→4  ✓  overlap: 'ing on the\nuse case.'


## Token mode

In [22]:
TOKEN_SIZE = 30
TOKEN_OVERLAP = 6

In [23]:
tok_chunker = SentenceChunker(
    ChunkParams(chunk_size=TOKEN_SIZE, chunk_overlap=TOKEN_OVERLAP, unit=ChunkUnit.tokens)
)
tok_chunks = tok_chunker.chunk(make_document(TEXT))

In [24]:
print(f"chunk_size={TOKEN_SIZE} tokens  chunk_overlap={TOKEN_OVERLAP} tokens")
print(f"Total chunks : {len(tok_chunks)}\n")

for c in tok_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}\u2013{c.char_end:<3}  tokens={c.token_count:>3}  \u2502 {c.text[:60]!r}")

chunk_size=30 tokens  chunk_overlap=6 tokens
Total chunks : 5

[0] chars   0–156  tokens= 30  │ 'Chunking is the process of splitting a long document into sm'
[1] chars 150–329  tokens= 36  │ 'model. The overlap\nensures that no information is lost at th'
[2] chars 323–535  tokens= 48  │ 'bours. Why should \nwe apply chunking? It is because without '
[3] chars 529–687  tokens= 34  │ ' step. The step is always smaller than the window, which cre'
[4] chars 681–687  tokens=  2  │ ' case.'


## Character Unit Comparison

In [25]:
print(f"{'Mode':<12} {'Chunks':>6}  {'Avg chars':>10}  {'Avg tokens':>10}")
print("-" * 44)

for label, chunks in [("characters", char_chunks), ("tokens", tok_chunks)]:
    avg_chars = sum(len(c.text) for c in chunks) / len(chunks)
    avg_tokens = sum(c.token_count for c in chunks) / len(chunks)
    print(f"{label:<12} {len(chunks):>6}  {avg_chars:>10.1f}  {avg_tokens:>10.1f}")

Mode         Chunks   Avg chars  Avg tokens
--------------------------------------------
characters        5       153.4        32.6
tokens            5       142.2        30.0
